# UnCNN — Model Architecture

### Imports & Libraries

The un-CNN pipeline consists of the weight seed, model architecture, and input preprocessing.

The model is **never trained** — weights are fixed random (seed 0) and the CNN is
used as a frozen feature extractor. The two preprocessing configs below are the
ones that produce the headline results (`DS_3ch_RobustScaler` for age regression,
`DS_3ch_RankSobel` for sex classification).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import nibabel as nib
import scipy.ndimage as ndi

# Fixed random weights for the (untrained) CNN
WEIGHT_SEED = 0

### Preprocessing
Volumes are resized to the MNI152 2mm grid (91 x 109 x 91). Each config builds a
3-channel input: intensity, local rank/median, and Sobel edge magnitude.

In [ ]:
def load_and_resize(path):
    img = nib.load(path).get_fdata().astype(np.float32)
    if img.shape != (91, 109, 91):
        img = ndi.zoom(img, [91 / img.shape[0], 109 / img.shape[1],
                              91 / img.shape[2]], order=1)
    return img


def clip_minmax(img, lo=1, hi=99):
    p_lo, p_hi = np.percentile(img, [lo, hi])
    img = np.clip(img, p_lo, p_hi)
    img_min, img_max = img.min(), img.max()
    return (img - img_min) / (img_max - img_min + 1e-6)


def norm(ch):
    mn, mx = ch.min(), ch.max()
    if mx - mn < 1e-8:
        return np.zeros_like(ch)
    return (ch - mn) / (mx - mn)


def sobel_mag(img):
    sx = ndi.sobel(img, axis=0)
    sy = ndi.sobel(img, axis=1)
    sz = ndi.sobel(img, axis=2)
    return np.sqrt(sx**2 + sy**2 + sz**2)


def rank_filter(img, size=3):
    ranked = ndi.percentile_filter(img, percentile=50, size=size)
    mn, mx = ranked.min(), ranked.max()
    if mx - mn < 1e-8:
        return np.zeros_like(ranked)
    return (ranked - mn) / (mx - mn)


def scale_robust(img, clip_low=2, clip_high=98, eps=1e-8):
    mask = img > np.percentile(img, 5)
    lo, hi = np.percentile(img[mask], [clip_low, clip_high])
    clipped = np.clip(img, lo, hi)
    brain = clipped[mask]
    q25, q75 = np.percentile(brain, [25, 75])
    iqr = float(q75 - q25)
    out = np.zeros_like(clipped, dtype=np.float32)
    out[mask] = (brain - float(np.median(brain))) / (iqr + eps)
    return out

In [ ]:
class DS_3ch_RobustScaler(Dataset):
    """Best overall age config."""
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = scale_robust(load_and_resize(self.paths[idx]))
        ranked = rank_filter(img, size=3)
        edges = norm(sobel_mag(img))
        return torch.from_numpy(np.stack([img, ranked, edges], axis=0)).float()


class DS_3ch_RankSobel(Dataset):
    """Best sex-accuracy config."""
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = clip_minmax(load_and_resize(self.paths[idx]), lo=2, hi=98)
        ranked = rank_filter(img, size=3)
        edges = norm(sobel_mag(img))
        return torch.from_numpy(np.stack([img, ranked, edges], axis=0)).float()

## Model

### Building block: depthwise separable 3D convolution

In [ ]:
class DepthwiseSepConv3d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv3d(in_ch, in_ch, kernel_size, padding=padding, groups=in_ch)
        self.pointwise = nn.Conv3d(in_ch, out_ch, 1)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))

### UnCNN

DoubleConv backbone with a covariance-pooling readout. The second-order
statistics (upper triangle of the channel covariance, concatenated with mean
features) capture channel co-activation patterns that mean pooling misses.
Features are pooled at all four depths and concatenated.

In [ ]:
class UnCNN(nn.Module):
    """DoubleConv + covariance pooling readout. Second-order statistics
    capture channel co-activation patterns that mean pooling misses."""
    def __init__(self, in_channels=3):
        super().__init__()
        self.conv1a = nn.Conv3d(in_channels, 64, 3, padding=1)
        self.norm1a = nn.GroupNorm(8, 64)
        self.conv1b = nn.Conv3d(64, 64, 3, padding=1)
        self.norm1b = nn.GroupNorm(8, 64)
        self.down1 = nn.AvgPool3d(2)

        self.conv2a = DepthwiseSepConv3d(64, 128)
        self.norm2a = nn.GroupNorm(8, 128)
        self.conv2b = DepthwiseSepConv3d(128, 128)
        self.norm2b = nn.GroupNorm(8, 128)
        self.down2 = nn.AvgPool3d(2)

        self.conv3a = DepthwiseSepConv3d(128, 256)
        self.norm3a = nn.GroupNorm(8, 256)
        self.conv3b = DepthwiseSepConv3d(256, 256)
        self.norm3b = nn.GroupNorm(8, 256)
        self.down3 = nn.AvgPool3d(2)

        self.conv4a = DepthwiseSepConv3d(256, 512)
        self.norm4a = nn.GroupNorm(8, 512)
        self.conv4b = DepthwiseSepConv3d(512, 512)
        self.norm4b = nn.GroupNorm(8, 512)
        self.down4 = nn.AvgPool3d(2)

        self.avg_pool = nn.AdaptiveAvgPool3d(2)

    def _cov_pool(self, x, max_ch=32):
        """Mean features + upper triangle of covariance matrix."""
        b, c = x.size(0), min(x.size(1), max_ch)
        mean_feats = self.avg_pool(x).view(b, -1)
        xc = x[:, :c]
        flat = xc.view(b, c, -1)
        flat = flat - flat.mean(dim=2, keepdim=True)
        cov = torch.bmm(flat, flat.transpose(1, 2)) / (flat.size(2) - 1)
        idx = torch.triu_indices(c, c, offset=1)
        cov_feats = cov[:, idx[0], idx[1]]
        return torch.cat([mean_feats, cov_feats], dim=1)

    def forward(self, x):
        x1 = F.relu(self.norm1a(self.conv1a(x)))
        x1 = self.down1(F.relu(self.norm1b(self.conv1b(x1))))
        x2 = F.relu(self.norm2a(self.conv2a(x1)))
        x2 = self.down2(F.relu(self.norm2b(self.conv2b(x2))))
        x3 = F.relu(self.norm3a(self.conv3a(x2)))
        x3 = self.down3(F.relu(self.norm3b(self.conv3b(x3))))
        x4 = F.relu(self.norm4a(self.conv4a(x3)))
        x4 = self.down4(F.relu(self.norm4b(self.conv4b(x4))))
        return torch.cat([
            self._cov_pool(x1, 32),
            self._cov_pool(x2, 32),
            self._cov_pool(x3, 32),
            self._cov_pool(x4, 32),
        ], dim=1)

### Instantiate with fixed seed

Seed the RNG before building the model so the fixed random weights are reproducible.


In [ ]:
torch.manual_seed(WEIGHT_SEED)
model = UnCNN(in_channels=3)